# Bước 10 – Chẩn đoán cách chia: phần thích nghi có giống phần đánh giá không?

Lý do làm bước này: ở bước 9, ngưỡng học từ phần thích nghi (10% luồng đầu của mỗi đợt) thường quá cao khi đem sang phần đánh giá (90% sau), dù mô hình vẫn xếp hạng tốt. Bước 10 kiểm tra xem hai phần có khác nhau không, và khác ở đâu.

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Chẩn đoán 4 mạng, in Bảng 10.1 đến 10.4 | 5–15 phút |

**Không cần GPU (card đồ họa).** Kết quả lưu vào `MyDrive/TADP_Q1/results/b10_chan_doan_chia/`.

Phiên bản mã nguồn này cũng bật **chế độ tất định** cho LightGBM (cây tăng cường gradient): cùng dữ liệu, cùng hạt giống ngẫu nhiên thì luôn ra cùng một mô hình, dù Colab cấp máy nào.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "diag_split.py": 10098,
            "experts.py": 5934, "few_label.py": 14044, "metrics.py": 4002, "preprocess.py": 3601, "shift.py": 3863}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 10" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 10


## Ô 2 – Chẩn đoán

In ra 4 dòng "[tên mạng] xong", rồi 4 bảng:
- Bảng 10.1: mỗi phần có bao nhiêu luồng, tỷ lệ tấn công, bao nhiêu loại tấn công, bao nhiêu đợt.
- Bảng 10.2: mỗi loại tấn công chiếm bao nhiêu phần trăm số luồng tấn công trong phần thích nghi và trong phần đánh giá.
- Bảng 10.3: hai mô hình, chấm trên từng phần. Mô hình A học trên toàn bộ nhãn của phần thích nghi. Mô hình B là mốc "đủ nhãn" của bước 9.
- Bảng 10.4: mô hình A theo từng loại tấn công.

Chạy xong báo Claude.

In [2]:
import sys, warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.diag_split import run_diag
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b10_chan_doan_chia",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_diag(cfg, data, splits, cols, pv="P1_log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
[CIC-IDS2017] xong trong 58 giây
[CSE-CIC-IDS2018] xong trong 54 giây
[CIC-ToN-IoT] xong trong 52 giây
[CIC-UNSW-NB15] xong trong 44 giây
BẢNG 10.1 – Mô tả từng phần
            mang      phan  so_dong  ty_le_tan_cong  so_loai_tan_cong  so_dot  so_ngay  ty_le_luong_trung_thoi_diem  so_dot_duoi_10_luong
    CIC-IDS2017     adapt    99972          0.2411                14      19        5                       0.3207                   NaN
    CIC-IDS2017      eval   899740          0.2411                15      20        5                       0.3462                   NaN
    CIC-IDS2017  eval_som   599827          0.2

(               mang       phan  so_dong  ty_le_tan_cong  so_loai_tan_cong  so_dot  so_ngay  ty_le_luong_trung_thoi_diem  so_dot_duoi_10_luong
 0       CIC-IDS2017      adapt    99972        0.241098                14      19        5                     0.320720                   NaN
 1       CIC-IDS2017       eval   899740        0.241087                15      20        5                     0.346227                   NaN
 2       CIC-IDS2017   eval_som   599827        0.241088                15      20        5                     0.341448                   NaN
 3       CIC-IDS2017  eval_muon   299913        0.241087                15      20        5                     0.352826                   NaN
 4       CIC-IDS2017    toan_bo   999712        0.241088                15      20        5                     0.343838                   3.0
 5   CSE-CIC-IDS2018      adapt    99981        0.055921                 9      22       11                     0.005121                   NaN